# Resume RAG Pipeline — v2

**What changed from v1:**
- Rich metadata per chunk: `candidate`, `section`, `page`, `filename`
- Docling `headings` field used to detect section automatically
- `filter_complex_metadata` strips nested dicts → fixes the Chroma `ValueError`
- Custom metadata injected **after** filtering, so `candidate` / `section` are never lost
- Metadata-filtered retrieval scoped to one candidate
- MMR retrieval to remove duplicate chunks
- Cross-encoder reranker as final step
- Stronger embedding model: `BAAI/bge-base-en-v1.5`
- Query parser to auto-extract candidate name + section from the user question

In [1]:
!pip install -q langchain langchain-community langchain-docling langchain-huggingface langchain-chroma chromadb sentence-transformers python-dotenv langchain-groq

In [2]:
!uv pip install langchain-groq python-dotenv

Using Python 3.14.5 environment at: C:\Users\rokyd\OneDrive\Desktop\resume checker\.venv
Checked 2 packages in 62ms


## Cell 2 — Imports

In [3]:
import re
from pathlib import Path
from collections import Counter

from langchain_docling import DoclingLoader
from langchain_core.documents import Document
from langchain_community.vectorstores.utils import filter_complex_metadata
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from sentence_transformers import CrossEncoder
from groq import Groq

c:\Users\rokyd\OneDrive\Desktop\resume checker\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\rokyd\AppData\Local\Temp\ipykernel_1904\633215315.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores.utils import filter_complex_metadata


## Cell 3 — Config

In [4]:
PDF_FOLDER   = Path("../pdf")
CHROMA_DIR   = "../chroma_db"
USE_MMR      = False    # Set to True when you want diversity-aware retrieval
SEARCH_K     = 5
FETCH_K      = 20

# Known section keywords (Docling headings are matched against these)
SECTION_KEYWORDS = [
    "profile", "summary", "objective",
    "experience", "work", "employment",
    "education", "academic",
    "projects", "project",
    "skills", "technical skills", "technologies",
    "certifications", "certificates",
    "achievements", "awards", "honors",
    "contact", "personal",
    "internship", "training",
    "publications", "research",
    "languages", "hobbies", "interests",
]

## Cell 4 — Load PDFs with DoclingLoader

In [5]:
pdf_files = sorted(PDF_FOLDER.glob("*.pdf"))

if not pdf_files:
    raise FileNotFoundError(f"No PDFs found in {PDF_FOLDER}")

print(f"Found {len(pdf_files)} PDF(s):")
for p in pdf_files:
    print(f"  • {p.name}")

loader = DoclingLoader(file_path=[str(p) for p in pdf_files])
raw_docs = loader.load()

print(f"\nTotal raw documents (chunks from Docling): {len(raw_docs)}")

Found 3 PDF(s):
  • Gourab Personal.pdf
  • Sukhendu Dutta.pdf
  • Syed Sadiqu Hussain.pdf


The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
[INFO] 2026-06-27 15:59:18,111 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-06-27 15:59:18,145 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\rokyd\OneDrive\Desktop\resume checker\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-06-27 15:59:18,147 [RapidOCR] main.py:63: Using C:\Users\rokyd\OneDrive\Desktop\resume checker\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-06-27 15:59:18,394 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-06-27 15:59:18,399 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\rokyd\OneDrive\Desktop\resume checker\.venv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-


Total raw documents (chunks from Docling): 29


## Cell 5 — Helper: detect section from Docling headings

In [6]:
def detect_section(headings: list[str]) -> str:
    """
    Match the nearest heading against known resume section keywords.
    Returns the matched keyword (title-cased) or 'General' if nothing matches.
    """
    for heading in headings:
        heading_lower = heading.lower().strip()
        for keyword in SECTION_KEYWORDS:
            if keyword in heading_lower:
                return keyword.title()  # e.g. "Projects", "Skills"
    return "General"

## Cell 6 — Build enriched docs with clean metadata

**Two-step approach:**
1. `filter_complex_metadata` strips the nested Docling dict that caused the `ValueError`
2. We then inject our own flat metadata: `candidate`, `section`, `page`, `filename`

In [7]:
def build_clean_docs(raw_docs: list[Document]) -> list[Document]:
    clean_docs = []

    for doc in raw_docs:
        # ── Step 1: grab what we need BEFORE filtering wipes it ──
        raw_meta = doc.metadata

        # Source path lives at 'source' or 'dl_meta' → 'origin' → 'filename'
        source_path = raw_meta.get("source", "")
        filename    = Path(source_path).name if source_path else "unknown.pdf"
        candidate   = Path(source_path).stem if source_path else "Unknown"

        # Docling puts headings in dl_meta (nested dict) before filtering
        dl_meta   = raw_meta.get("dl_meta", {})
        headings  = dl_meta.get("headings", []) if isinstance(dl_meta, dict) else []
        section   = detect_section(headings)

        # Page number
        page = 1  # default
        if isinstance(dl_meta, dict):
            doc_items = dl_meta.get("doc_items", [])
            if doc_items and isinstance(doc_items[0], dict):
                provs = doc_items[0].get("prov", [])
                if provs and isinstance(provs[0], dict):
                    page = provs[0].get("page_no", 1)

        # ── Step 2: strip ALL complex metadata (nested dicts, lists of dicts) ──
        flat_docs = filter_complex_metadata([doc])
        flat_doc  = flat_docs[0] if flat_docs else doc

        # ── Step 3: inject our own clean, flat metadata ──
        flat_doc.metadata["candidate"] = candidate   # e.g. "Sukhendu Dutta"
        flat_doc.metadata["filename"]  = filename    # e.g. "Sukhendu Dutta.pdf"
        flat_doc.metadata["section"]   = section     # e.g. "Projects"
        flat_doc.metadata["page"]      = int(page)   # e.g. 1

        if flat_doc.page_content.strip():
            clean_docs.append(flat_doc)

    return clean_docs


clean_docs = build_clean_docs(raw_docs)

print(f"Clean docs ready: {len(clean_docs)}")
print("\nSample metadata:", clean_docs[0].metadata)

Clean docs ready: 29

Sample metadata: {'source': '..\\pdf\\Gourab Personal.pdf', 'candidate': 'Gourab Personal', 'filename': 'Gourab Personal.pdf', 'section': 'Profile', 'page': 1}


## Cell 7 — Verify metadata distribution

In [8]:
print("Chunks per candidate:")
for name, count in Counter(d.metadata["candidate"] for d in clean_docs).items():
    print(f"  {name}: {count}")

print("\nChunks per section:")
for section, count in Counter(d.metadata["section"] for d in clean_docs).most_common():
    print(f"  {section}: {count}")

Chunks per candidate:
  Gourab Personal: 9
  Sukhendu Dutta: 10
  Syed Sadiqu Hussain: 10

Chunks per section:
  General: 14
  Profile: 2
  Education: 2
  Skills: 2
  Work: 2
  Achievements: 2
  Experience: 1
  Objective: 1
  Projects: 1
  Hobbies: 1
  Personal: 1


## Cell 8 — Embedding model

`BAAI/bge-base-en-v1.5` separates semantically similar resume text better than `all-MiniLM-L6-v2`.

In [9]:
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5",
    encode_kwargs={"normalize_embeddings": True},  # required for BGE cosine similarity
)

print("Embedding model loaded")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3094.23it/s]


Embedding model loaded


## Cell 9 — Build Chroma VectorStore

In [10]:
print("Indexing into Chroma ...")

vector_store = Chroma.from_documents(
    documents=clean_docs,
    embedding=embedding_model,
    persist_directory=CHROMA_DIR,
)

print(f"Done. Total vectors: {vector_store._collection.count()}")

Indexing into Chroma ...
Done. Total vectors: 116


## Cell 10 — (Optional) Reload from disk on subsequent runs

In [11]:
# Uncomment to skip re-embedding on every run
# vector_store = Chroma(
#     persist_directory=CHROMA_DIR,
#     embedding_function=embedding_model,
# )
# print("Loaded from disk:", vector_store._collection.count(), "vectors")

## Cell 11 — Query parser

Extracts candidate name and section keyword from the user's question.

In [12]:
def parse_query(query: str, known_candidates: list[str]) -> dict:
    """
    Returns {"candidate": str | None, "section": str | None}
    """
    query_lower = query.lower()

    # --- Candidate detection ---
    detected_candidate = None
    for name in known_candidates:
        # Match on full name or first name
        first_name = name.split()[0].lower()
        if name.lower() in query_lower or first_name in query_lower:
            detected_candidate = name
            break

    # --- Section detection ---
    detected_section = None
    for keyword in SECTION_KEYWORDS:
        if keyword in query_lower:
            detected_section = keyword.title()
            break

    return {"candidate": detected_candidate, "section": detected_section}


# All known candidate names from indexed docs
known_candidates = list({d.metadata["candidate"] for d in clean_docs})
print("Known candidates:", known_candidates)

Known candidates: ['Sukhendu Dutta', 'Syed Sadiqu Hussain', 'Gourab Personal']


## Cell 12 — Similarity Retriever with Metadata Filter

In [18]:
def get_retriever(candidate: str = None, section: str = None):
    """
    Returns a standard similarity retriever filtered to candidate and/or section.
    """
    chroma_filter = {}

    if candidate and section:
        chroma_filter = {"$and": [
            {"candidate": {"$eq": candidate}},
            {"section":   {"$eq": section}},
        ]}
    elif candidate:
        chroma_filter = {"candidate": {"$eq": candidate}}
    elif section:
        chroma_filter = {"section": {"$eq": section}}

    search_kwargs = {
        "k": SEARCH_K,
    }
    if chroma_filter:
        search_kwargs["filter"] = chroma_filter

    return vector_store.as_retriever(
        search_type="similarity",
        search_kwargs=search_kwargs,
    )


def search(query: str, candidate: str = None, section: str = None):
    """
    Retrieve raw documents via the same similarity retriever used by the pipeline.
    """
    retriever = get_retriever(candidate=candidate, section=section)
    return retriever.invoke(query)

## Cell 13 — Cross-encoder reranker

In [19]:
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(query: str, docs: list[Document], top_n: int = 3) -> list[Document]:
    if not docs:
        return []
    pairs  = [(query, doc.page_content) for doc in docs]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(scores, docs), key=lambda x: x[0], reverse=True)
    return [doc for _, doc in ranked[:top_n]]

print("Reranker ready")

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 6036.38it/s]


Reranker ready


## Cell 14 — Groq Llama completion

In [20]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from dotenv import load_dotenv

load_dotenv()

# Initialize your Groq LLM
llm = ChatGroq(
    model="meta-llama/llama-4-scout-17b-16e-instruct",
    temperature=0.2,
    max_tokens=1024,
)

# Build a clean system prompt to pass the retrieved chunks
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "You are an expert HR assistant. Answer the user's question accurately using only "
        "the provided resume context. If you don't know the answer or if the context is "
        "empty, say that the information was not found.\n\n"
        "Retrieved Resume Context:\n{context}"
    )),
    ("human", "{question}"),
])

print("Groq LLM and Prompt Template ready")

Groq LLM and Prompt Template ready


## Cell 15 — Full pipeline: query → parse → filter → MMR → rerank

In [21]:
def answer_question(query: str, top_n: int = 3, verbose: bool = True) -> str:
    """
    End-to-end RAG workflow:
    1. Parse query for metadata tags.
    2. Fetch via standard similarity search.
    3. Rerank using CrossEncoder.
    4. Synthesize final response using Groq LLM.
    """
    # 1. Parsing
    parsed = parse_query(query, known_candidates)
    candidate = parsed["candidate"]
    section   = parsed["section"]

    if verbose:
        print(f"Query    : {query}")
        print(f"Candidate: {candidate}")
        print(f"Section  : {section}")

    # 2. Retrieval
    retriever = get_retriever(candidate=candidate, section=section)
    retrieved = retriever.invoke(query)
    
    if verbose:
        print(f"Retrieved {len(retrieved)} chunks via similarity search.")

    # 3. Reranking
    reranked_docs = rerank(query, retrieved, top_n=top_n)
    
    # 4. Context Preparation
    context_str = ""
    for idx, doc in enumerate(reranked_docs, 1):
        m = doc.metadata
        context_str += f"\n--- Document Chunk {idx} (Candidate: {m['candidate']}, Section: {m['section']}) ---\n"
        context_str += f"{doc.page_content}\n"
        
    if not context_str.strip():
        context_str = "No relevant chunks found matching your metadata criteria."

    # 5. Generation
    chain = rag_prompt | llm
    response = chain.invoke({"context": context_str, "question": query})
    
    return response.content

## Cell 16 — Test queries

In [22]:
# ── Test 1: candidate + section filter ──
answer = answer_question(query = "Give me details about Sukhendu Dutta's Projects")

print("=" * 60)
print(answer)


Query    : Give me details about Sukhendu Dutta's Projects
Candidate: Sukhendu Dutta
Section  : Projects
Retrieved 0 chunks via similarity search.
The information was not found.


In [23]:
# ── Test 2: candidate only (no section) ──
results = search("What are Gourab's skills and work experience?")

print("=" * 60)
for i, doc in enumerate(results, 1):
    m = doc.metadata
    print(f"\n── Result {i} | candidate: {m['candidate']} | section: {m['section']} ──")
    print(doc.page_content[:400])


── Result 1 | candidate: Gourab Personal | section: General ──
GOURAB KUMAR SAHOO
Balasore, Odisha
+91-9938348522 gourabkumarsahoo083@gmail.com Linkedin Github
-Feb 2026

── Result 2 | candidate: Gourab Personal | section: General ──
GOURAB KUMAR SAHOO
Balasore, Odisha
+91-9938348522 gourabkumarsahoo083@gmail.com Linkedin Github
-Feb 2026

── Result 3 | candidate: Gourab Personal | section: General ──
GOURAB KUMAR SAHOO
Balasore, Odisha
+91-9938348522 gourabkumarsahoo083@gmail.com Linkedin Github
-Feb 2026

── Result 4 | candidate: Gourab Personal | section: General ──
GOURAB KUMAR SAHOO
Balasore, Odisha
+91-9938348522 gourabkumarsahoo083@gmail.com Linkedin Github
-Feb 2026

── Result 5 | candidate: Gourab Personal | section: Profile ──
PROFILE
I am an enthusiastic final year Computer Science student equipped with strong problem-solving skills and practical experience in frontend web development. I specialize in creating responsive and user-centric interfaces using modern technologies

In [24]:
# ── Test 3: section only (no candidate) → all resumes ──
results = search("List all certifications across all resumes")

print("=" * 60)
for i, doc in enumerate(results, 1):
    m = doc.metadata
    print(f"\n── Result {i} | candidate: {m['candidate']} | section: {m['section']} ──")
    print(doc.page_content[:400])


── Result 1 | candidate: Syed Sadiqu Hussain | section: General ──
TECHNICAL SKILL
- Languages -Python, Java, Dart
- Frameworks/Libraries -FastAPI, Spring Boot, Spring Data JPA, Flutter
- Database -MySQL, PostgreSQL, MongoDB
- Cloud Services -Azure
- Developer tools -Git/GitHub, Docker, Postman, Firebase

── Result 2 | candidate: Syed Sadiqu Hussain | section: General ──
TECHNICAL SKILL
- Languages -Python, Java, Dart
- Frameworks/Libraries -FastAPI, Spring Boot, Spring Data JPA, Flutter
- Database -MySQL, PostgreSQL, MongoDB
- Cloud Services -Azure
- Developer tools -Git/GitHub, Docker, Postman, Firebase

── Result 3 | candidate: Syed Sadiqu Hussain | section: General ──
TECHNICAL SKILL
- Languages -Python, Java, Dart
- Frameworks/Libraries -FastAPI, Spring Boot, Spring Data JPA, Flutter
- Database -MySQL, PostgreSQL, MongoDB
- Cloud Services -Azure
- Developer tools -Git/GitHub, Docker, Postman, Firebase

── Result 4 | candidate: Syed Sadiqu Hussain | section: General ──
TECHNICAL S

In [25]:
# ── Test 4: direct filter (bypass parser) ──
retriever = get_retriever(candidate="Sukhendu Dutta", section="Skills")
results   = retriever.invoke("python machine learning")

print(f"Direct filter results: {len(results)}")
for doc in results:
    print(f"  [{doc.metadata['section']}] {doc.page_content[:200]}")
    print()

Direct filter results: 4
  [Skills] TECHNICAL SKILLS
Languages:
Python, SQL, Java
AI/ML:
Scikit-learn, Pandas, NumPy, Matplotlib, Feature Engineering, Statistical Modeling
Web:
HTML5, CSS3, JavaScript
Tools:
Git, GitHub, Docker, VS Code

  [Skills] TECHNICAL SKILLS
Languages:
Python, SQL, Java
AI/ML:
Scikit-learn, Pandas, NumPy, Matplotlib, Feature Engineering, Statistical Modeling
Web:
HTML5, CSS3, JavaScript
Tools:
Git, GitHub, Docker, VS Code

  [Skills] TECHNICAL SKILLS
Languages:
Python, SQL, Java
AI/ML:
Scikit-learn, Pandas, NumPy, Matplotlib, Feature Engineering, Statistical Modeling
Web:
HTML5, CSS3, JavaScript
Tools:
Git, GitHub, Docker, VS Code

  [Skills] TECHNICAL SKILLS
Languages:
Python, SQL, Java
AI/ML:
Scikit-learn, Pandas, NumPy, Matplotlib, Feature Engineering, Statistical Modeling
Web:
HTML5, CSS3, JavaScript
Tools:
Git, GitHub, Docker, VS Code

